In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
from datetime import datetime


# Loading Data

In [13]:
# Load datasets
print(f"Starting process at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Loading existing features and batting data...")

# Load features dataset
features_df = pd.read_csv(r'/data/processed_data/all_league_bowl_features.csv')
features_df['date'] = pd.to_datetime(features_df['date'])

# Load original batting data
bowling_df = pd.read_csv(r'/data/mbm/all_league_bowl_final.csv')
bowling_df['date'] = pd.to_datetime(bowling_df['date'])


Starting process at 2025-04-13 18:52:49
Loading existing features and batting data...


In [14]:
bowling_df.drop(columns=['Unnamed: 0'], inplace=True)

# Filling NULL values for league debuts

In [32]:
def fill_all_bowling_nulls(features_df, bowling_df):
    """
    Fill NULL values in bowling features dataframe for:
    1. League debuts (where career stats are NULL)
    2. Season debuts (where season stats are NULL)
    3. Venue debuts (where venue stats are NULL)
    4. Innings debuts (where innings stats are NULL)
    """
    # Create a copy to avoid modifying the original
    features_df = features_df.copy()

    # Ensure proper date format
    bowling_df['date'] = pd.to_datetime(bowling_df['date'])
    features_df['date'] = pd.to_datetime(features_df['date'])

    # Pre-process bowling data once
    print("Pre-processing bowling data...")
    bowling_df = bowling_df.copy()

    # Add season column
    bowling_df['season'] = bowling_df['date'].dt.year

    # Convert overs to balls
    bowling_df['overs'] = bowling_df['overs'].replace('DNB', '0.0').astype(str)
    bowling_df[['overs_int', 'balls_int']] = bowling_df['overs'].str.split('.', expand=True).astype(int)
    bowling_df['total_balls'] = bowling_df['overs_int'] * 6 + bowling_df['balls_int']
    bowling_df['total_overs'] = bowling_df['total_balls'] / 6

    # Add dot balls column if it doesn't exist
    if 'dots' not in bowling_df.columns:
        bowling_df['dots'] = 0

    # Get unique player IDs
    unique_players = features_df['bowler_id'].unique()
    print(f"Processing {len(unique_players)} unique players")

    # Counters for tracking updates
    updates_made = 0
    null_cases = {'league_debuts': 0, 'season_debuts': 0, 'venue_debuts': 0, 'innings_debuts': 0}

    # Process each player
    for i, player_id in enumerate(unique_players):
        if (i+1) % 100 == 0 or i+1 == len(unique_players):
            print(f"Processed {i+1}/{len(unique_players)} players")

        # Get player data
        player_features = features_df[features_df['bowler_id'] == player_id]

        # Skip if player only has one entry (absolute debut)
        if len(player_features) <= 1:
            continue

        # Get player's first match date
        first_match_date = player_features['date'].min()

        # Find rows with NULL values in any of the categories
        null_careers = player_features[
            (pd.isna(player_features['career_avg'])) &
            (player_features['date'] > first_match_date)
        ]

        null_seasons = player_features[
            pd.isna(player_features['ssn_avg']) &
            ~pd.isna(player_features['career_avg'])  # Only include rows where career is NOT null
        ]

        null_venues = player_features[
            pd.isna(player_features['venue_avg']) &
            ~pd.isna(player_features['career_avg'])
        ]

        null_innings = player_features[
            pd.isna(player_features['inns_avg']) &
            ~pd.isna(player_features['career_avg'])
        ]

        # Update counters
        null_cases['league_debuts'] += len(null_careers)
        null_cases['season_debuts'] += len(null_seasons)
        null_cases['venue_debuts'] += len(null_venues)
        null_cases['innings_debuts'] += len(null_innings)

        # Combine all null rows for efficient processing
        all_null_rows = pd.concat([null_careers, null_seasons, null_venues, null_innings]).drop_duplicates()

        if len(all_null_rows) == 0:
            continue

        # Get bowling data for this player and sort by date
        player_bowling = bowling_df[bowling_df['bowler_id'] == player_id].sort_values('date')

        # Process each null row
        for idx, null_row in all_null_rows.iterrows():
            current_date = null_row['date']
            current_league = null_row['league']
            current_season = null_row['season']
            current_venue = null_row['venue']
            current_inns = null_row['inns']

            # Get all matches before this date
            prior_matches = player_bowling[player_bowling['date'] < current_date]

            if len(prior_matches) == 0:
                continue

            updates_made += 1

            # Calculate aggregated stats once
            total_wickets = prior_matches['wickets'].sum()
            total_runs = prior_matches['runs'].sum()
            total_matches = len(prior_matches)
            total_balls = prior_matches['total_balls'].sum()
            total_overs = prior_matches['total_overs'].sum()
            total_dots = prior_matches['dots'].sum()

            # Career stats
            career_avg = total_runs / max(total_wickets, 1)
            career_eco = total_runs / max(total_overs, 1)
            career_sr = total_balls / max(total_wickets, 1)
            career_wpm = total_wickets / max(total_matches, 1)
            career_opm = total_overs / max(total_matches, 1)
            career_dotpct = total_dots / max(total_balls, 1)

            # FILL CAREER STATS IF NEEDED
            if pd.isna(features_df.loc[idx, 'career_avg']):
                features_df.loc[idx, 'career_avg'] = career_avg
                features_df.loc[idx, 'career_eco'] = career_eco
                features_df.loc[idx, 'career_sr'] = career_sr
                features_df.loc[idx, 'career_wpm'] = career_wpm
                features_df.loc[idx, 'career_opm'] = career_opm

                if 'career_dotpct' in features_df.columns:
                    features_df.loc[idx, 'career_dotpct'] = career_dotpct

            # FILL INNINGS STATS IF NEEDED
            if pd.isna(features_df.loc[idx, 'inns_avg']):
                # Try to get innings-specific prior matches
                prior_inns_matches = prior_matches[prior_matches['inns'] == current_inns]

                if len(prior_inns_matches) > 0:
                    # Use innings-specific stats
                    inns_wickets = prior_inns_matches['wickets'].sum()
                    inns_runs = prior_inns_matches['runs'].sum()
                    inns_matches = len(prior_inns_matches)
                    inns_balls = prior_inns_matches['total_balls'].sum()
                    inns_overs = prior_inns_matches['total_overs'].sum()
                    inns_dots = prior_inns_matches['dots'].sum()

                    inns_avg = inns_runs / max(inns_wickets, 1)
                    inns_eco = inns_runs / max(inns_overs, 1)
                    inns_sr = inns_balls / max(inns_wickets, 1)
                    inns_wpm = inns_wickets / max(inns_matches, 1)
                    inns_opm = inns_overs / max(inns_matches, 1)
                    inns_dotpct = inns_dots / max(inns_balls, 1)
                else:
                    # Fallback to career stats
                    inns_avg = career_avg
                    inns_eco = career_eco
                    inns_sr = career_sr
                    inns_wpm = career_wpm
                    inns_opm = career_opm
                    inns_dotpct = career_dotpct

                features_df.loc[idx, 'inns_avg'] = inns_avg
                features_df.loc[idx, 'inns_eco'] = inns_eco
                features_df.loc[idx, 'inns_sr'] = inns_sr
                features_df.loc[idx, 'inns_wpm'] = inns_wpm
                features_df.loc[idx, 'inns_opm'] = inns_opm

                if 'inns_dotpct' in features_df.columns:
                    features_df.loc[idx, 'inns_dotpct'] = inns_dotpct

            # FILL VENUE STATS IF NEEDED
            if pd.isna(features_df.loc[idx, 'venue_avg']):
                # Try to get venue-specific prior matches
                prior_venue_matches = prior_matches[prior_matches['venue'] == current_venue]

                if len(prior_venue_matches) > 0:
                    # Use venue-specific stats
                    venue_wickets = prior_venue_matches['wickets'].sum()
                    venue_runs = prior_venue_matches['runs'].sum()
                    venue_matches = len(prior_venue_matches)
                    venue_balls = prior_venue_matches['total_balls'].sum()
                    venue_overs = prior_venue_matches['total_overs'].sum()
                    venue_dots = prior_venue_matches['dots'].sum()

                    venue_avg = venue_runs / max(venue_wickets, 1)
                    venue_eco = venue_runs / max(venue_overs, 1)
                    venue_sr = venue_balls / max(venue_wickets, 1)
                    venue_wpm = venue_wickets / max(venue_matches, 1)
                    venue_opm = venue_overs / max(venue_matches, 1)
                    venue_dotpct = venue_dots / max(venue_balls, 1)
                else:
                    # Fallback to career stats
                    venue_avg = career_avg
                    venue_eco = career_eco
                    venue_sr = career_sr
                    venue_wpm = career_wpm
                    venue_opm = career_opm
                    venue_dotpct = career_dotpct

                features_df.loc[idx, 'venue_avg'] = venue_avg
                features_df.loc[idx, 'venue_eco'] = venue_eco
                features_df.loc[idx, 'venue_sr'] = venue_sr
                features_df.loc[idx, 'venue_wpm'] = venue_wpm
                features_df.loc[idx, 'venue_opm'] = venue_opm

                if 'venue_dotpct' in features_df.columns:
                    features_df.loc[idx, 'venue_dotpct'] = venue_dotpct

            # FILL SEASON STATS IF NEEDED
            if pd.isna(features_df.loc[idx, 'ssn_avg']):
                # Strategy 1: Current season in other leagues
                current_season_other_leagues = prior_matches[
                    (prior_matches['season'] == current_season) &
                    (prior_matches['league'] != current_league)
                ]

                # Strategy 2: Previous season in same league
                prior_same_league = prior_matches[prior_matches['league'] == current_league]

                if len(current_season_other_leagues) > 0:
                    # Use other leagues in current season
                    season_wickets = current_season_other_leagues['wickets'].sum()
                    season_runs = current_season_other_leagues['runs'].sum()
                    season_matches = len(current_season_other_leagues)
                    season_balls = current_season_other_leagues['total_balls'].sum()
                    season_overs = current_season_other_leagues['total_overs'].sum()
                    season_dots = current_season_other_leagues['dots'].sum()

                elif len(prior_same_league) > 0:
                    # Try to get most recent previous season
                    seasons = sorted(prior_same_league['season'].unique())
                    prev_seasons = [s for s in seasons if s < current_season]

                    if prev_seasons:
                        prev_season = max(prev_seasons)
                        prev_season_matches = prior_same_league[prior_same_league['season'] == prev_season]

                        season_wickets = prev_season_matches['wickets'].sum()
                        season_runs = prev_season_matches['runs'].sum()
                        season_matches = len(prev_season_matches)
                        season_balls = prev_season_matches['total_balls'].sum()
                        season_overs = prev_season_matches['total_overs'].sum()
                        season_dots = prev_season_matches['dots'].sum()
                    else:
                        # Fall back to all prior matches
                        season_wickets = total_wickets
                        season_runs = total_runs
                        season_matches = total_matches
                        season_balls = total_balls
                        season_overs = total_overs
                        season_dots = total_dots
                else:
                    # Fall back to all prior matches
                    season_wickets = total_wickets
                    season_runs = total_runs
                    season_matches = total_matches
                    season_balls = total_balls
                    season_overs = total_overs
                    season_dots = total_dots

                # Calculate and update season stats
                ssn_avg = season_runs / max(season_wickets, 1)
                ssn_eco = season_runs / max(season_overs, 1)
                ssn_sr = season_balls / max(season_wickets, 1)
                ssn_wpm = season_wickets / max(season_matches, 1)
                ssn_opm = season_overs / max(season_matches, 1)
                ssn_dotpct = season_dots / max(season_balls, 1)

                features_df.loc[idx, 'ssn_avg'] = ssn_avg
                features_df.loc[idx, 'ssn_eco'] = ssn_eco
                features_df.loc[idx, 'ssn_sr'] = ssn_sr
                features_df.loc[idx, 'ssn_wpm'] = ssn_wpm
                features_df.loc[idx, 'ssn_opm'] = ssn_opm

                if 'ssn_dotpct' in features_df.columns:
                    features_df.loc[idx, 'ssn_dotpct'] = ssn_dotpct

    # Print summary
    print(f"\nSummary of NULL value fills:")
    for case_type, count in null_cases.items():
        print(f"- {case_type}: {count}")
    print(f"Total updates made: {updates_made}")

    return features_df

In [33]:
updated_features_df = fill_all_bowling_nulls(features_df, bowling_df)

Pre-processing bowling data...
Processing 4691 unique players
Processed 100/4691 players
Processed 200/4691 players
Processed 300/4691 players
Processed 400/4691 players
Processed 500/4691 players
Processed 600/4691 players
Processed 700/4691 players
Processed 800/4691 players
Processed 900/4691 players
Processed 1000/4691 players
Processed 1100/4691 players
Processed 1200/4691 players
Processed 1300/4691 players
Processed 1400/4691 players
Processed 1500/4691 players
Processed 1600/4691 players
Processed 1700/4691 players
Processed 1800/4691 players
Processed 1900/4691 players
Processed 2000/4691 players
Processed 2100/4691 players
Processed 2200/4691 players
Processed 2300/4691 players
Processed 2400/4691 players
Processed 2500/4691 players
Processed 2600/4691 players
Processed 2700/4691 players
Processed 2800/4691 players
Processed 2900/4691 players
Processed 3000/4691 players
Processed 3100/4691 players
Processed 3200/4691 players
Processed 3300/4691 players
Processed 3400/4691 pla

# Checking for NULL filled values

In [21]:
updated_features_df.sample()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,venue,season,league,...,ssn_avg,ssn_eco,ssn_sr,ssn_wpm,ssn_opm,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm
117755,2017-07-30,1068369,Worcestershire,Durham,288617,RA Whiteley,1,County Ground,2017,T20BLAST,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
# Find a player with NULL values across leagues for demonstration
example_player_id = 625383  # Example fallback (Jasprit Bumrah)
print(f"Checking for player ID {example_player_id} or finding a suitable example...")

player_data = features_df[features_df['bowler_id'] == example_player_id].sort_values('date')

if player_data.empty or player_data['career_avg'].isna().sum() <= 1:
    # Find a player who has played in multiple leagues and has NULL values
    league_counts = features_df.groupby('bowler_id')['league'].nunique()
    multi_league_players = league_counts[league_counts > 1].index.tolist()

    for pid in multi_league_players:
        p_data = features_df[features_df['bowler_id'] == pid].sort_values('date')
        null_counts = p_data['career_avg'].isna().sum()

        if null_counts > 1:  # More than just the first match
            example_player_id = pid
            player_data = p_data
            player_name = player_data['bowler'].iloc[0] if 'bowler' in player_data.columns else f"ID: {pid}"
            print(f"Found example player: {player_name} (ID: {pid})")
            break

# Get the leagues the player has played in
player_leagues = player_data['league'].unique()

# Find a league debut (not the first appearance overall)
first_date = player_data['date'].min()
league_debuts = player_data[pd.isna(player_data['career_avg']) & (player_data['date'] > first_date)]

if not league_debuts.empty:
    debut_idx = league_debuts.index[0]
    debut_league = league_debuts.iloc[0]['league']
    debut_date = league_debuts.iloc[0]['date']

    # Display before filling
    print("\n--- BEFORE FILLING ---")
    print(f"Row with NULL value (first {debut_league} match):")
    display(features_df.loc[[debut_idx], ['date', 'league', 'career_avg', 'career_eco', 'career_sr', 'career_wpm', 'career_opm']])

    # Show prior data from other leagues
    prior_leagues = player_data[(player_data['date'] < debut_date) & (player_data['league'] != debut_league)]
    if not prior_leagues.empty:
        print(f"\nPrior data from {prior_leagues.iloc[0]['league']} before {debut_league} debut:")
        display(prior_leagues[['date', 'league', 'career_avg', 'career_eco', 'career_sr', 'career_wpm', 'career_opm']])

    # Display after filling
    print("\n--- AFTER FILLING ---")
    print(f"Row with filled value (first {debut_league} match):")
    display(updated_features_df.loc[[debut_idx], ['date', 'league', 'career_avg', 'career_eco', 'career_sr', 'career_wpm', 'career_opm']])

    # Also check venue stats
    print("\n--- VENUE STATS COMPARISON ---")
    print("Before filling:")
    display(features_df.loc[[debut_idx], ['date', 'venue', 'venue_avg', 'venue_eco', 'venue_sr', 'venue_wpm', 'venue_opm']])
    print("After filling:")
    display(updated_features_df.loc[[debut_idx], ['date', 'venue', 'venue_avg', 'venue_eco', 'venue_sr', 'venue_wpm', 'venue_opm']])

    # Check season stats
    print("\n--- SEASON STATS COMPARISON ---")
    print("Before filling:")
    display(features_df.loc[[debut_idx], ['date', 'season', 'ssn_avg', 'ssn_eco', 'ssn_sr', 'ssn_wpm', 'ssn_opm']])
    print("After filling:")
    display(updated_features_df.loc[[debut_idx], ['date', 'season', 'ssn_avg', 'ssn_eco', 'ssn_sr', 'ssn_wpm', 'ssn_opm']])

    print("\n✅ Success! The NULL values have been filled with the bowler's stats from prior leagues.")
else:
    print("❌ Could not find a suitable example of a league debut for demonstration.")

# Example usage:
# updated_features_df = fill_league_debut_nulls_bowling(features_df, bowling_df)

Checking for player ID 625383 or finding a suitable example...

--- BEFORE FILLING ---
Row with NULL value (first SMA match):


,date,league,career_avg,career_eco,career_sr,career_wpm,career_opm
63841,2016-01-02,SMA,NaN,NaN,NaN,NaN,NaN



Prior data from IPL before SMA debut:


,date,league,career_avg,career_eco,career_sr,career_wpm,career_opm
63824,2013-04-04,IPL,NaN,NaN,NaN,NaN,NaN
63825,2013-04-21,IPL,10.666667,8.000000,8.000000,3.000000,4.000000
63826,2014-04-19,IPL,23.333333,10.000000,14.000000,1.500000,3.500000
63827,2014-05-03,IPL,28.000000,9.333333,18.000000,1.000000,3.000000
63828,2014-05-06,IPL,36.333333,8.384615,26.000000,0.750000,3.250000
63829,2014-05-10,IPL,26.200000,7.705882,20.400000,1.000000,3.400000
63830,2014-05-12,IPL,32.800000,7.809524,25.200000,0.833333,3.500000
63831,2014-05-14,IPL,37.600000,7.520000,30.000000,0.714286,3.571429
63832,2014-05-19,IPL,42.200000,7.535714,33.600000,0.625000,3.500000
63833,2014-05-21,IPL,46.800000,7.548387,37.200000,0.555556,3.444444



--- AFTER FILLING ---
Row with filled value (first SMA match):


,date,league,career_avg,career_eco,career_sr,career_wpm,career_opm
63841,2016-01-02,SMA,50.454545,9.02439,33.545455,0.647059,3.617647


... (5 outputs left out)

,date,season,ssn_avg,ssn_eco,ssn_sr,ssn_wpm,ssn_opm
63841,2016-01-02,2016,NaN,NaN,NaN,NaN,NaN


After filling:


,date,season,ssn_avg,ssn_eco,ssn_sr,ssn_wpm,ssn_opm
63841,2016-01-02,2016,50.454545,9.02439,33.545455,0.647059,3.617647



✅ Success! The NULL values have been filled with the bowler's stats from prior leagues.


In [37]:
bowling_df[bowling_df.bowler_id== 625383]

,match_id,date,inns,bowler,bowler_id,bowl_team,bat_team,overs,runs,wickets,maidens,eco,balls,dots,lbw_or_bowled,venue,league
15921,597999,2013-04-04,1,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,32.0,3,0,8.000000,24,12,2,M.Chinnaswamy Stadium,IPL
16481,598024,2013-04-21,2,JJ Bumrah,625383,Mumbai Indians,Delhi Capitals,3.0,38.0,0,0,12.666667,21,5,0,Feroz Shah Kotla,IPL
24739,729287,2014-04-19,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,2.0,14.0,0,0,7.000000,12,4,0,Dubai International Cricket Stadium,IPL
25155,733973,2014-05-03,1,JJ Bumrah,625383,Mumbai Indians,Punjab Kings,4.0,25.0,0,0,6.250000,27,13,0,Wankhede Stadium,IPL
25276,733983,2014-05-06,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,22.0,2,0,5.500000,24,13,2,Wankhede Stadium,IPL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151096,1426286,2024-04-30,2,JJ Bumrah,625383,Mumbai Indians,Lucknow Super Giants,4.0,17.0,0,0,4.250000,24,16,0,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,IPL
151156,1426289,2024-05-03,1,JJ Bumrah,625383,Mumbai Indians,Kolkata Knight Riders,3.5,18.0,3,0,4.695652,23,15,2,Wankhede Stadium,IPL
151255,1426293,2024-05-06,1,JJ Bumrah,625383,Mumbai Indians,Sunrisers Hyderabad,4.0,23.0,1,0,5.750000,25,9,0,Wankhede Stadium,IPL
151372,1426298,2024-05-11,1,JJ Bumrah,625383,Mumbai Indians,Kolkata Knight Riders,4.0,39.0,2,0,9.750000,24,10,1,Eden Gardens,IPL


In [35]:
updated_features_df[updated_features_df.bowler_id== 625383]

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,venue,season,league,...,ssn_avg,ssn_eco,ssn_sr,ssn_wpm,ssn_opm,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm
63824,2013-04-04,597999,Mumbai Indians,Royal Challengers Bangalore,625383,JJ Bumrah,1,M.Chinnaswamy Stadium,2013,IPL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63825,2013-04-21,598024,Mumbai Indians,Delhi Capitals,625383,JJ Bumrah,2,Feroz Shah Kotla,2013,IPL,...,10.666667,8.000000,8.000000,3.000000,4.000000,10.666667,8.000000,8.000000,3.00,4.000000
63826,2014-04-19,729287,Mumbai Indians,Royal Challengers Bangalore,625383,JJ Bumrah,2,Dubai International Cricket Stadium,2014,IPL,...,23.333333,10.000000,14.000000,1.500000,3.500000,23.333333,10.000000,15.000000,1.50,3.500000
63827,2014-05-03,733973,Mumbai Indians,Punjab Kings,625383,JJ Bumrah,1,Wankhede Stadium,2014,IPL,...,0.000000,7.000000,0.000000,0.000000,2.000000,28.000000,9.333333,19.000000,1.00,3.000000
63828,2014-05-06,733983,Mumbai Indians,Royal Challengers Bangalore,625383,JJ Bumrah,2,Wankhede Stadium,2014,IPL,...,0.000000,6.500000,0.000000,0.000000,3.000000,36.333333,8.384615,28.000000,0.75,3.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64029,2024-06-22,1415747,India,Bangladesh,625383,JJ Bumrah,2,Sir Vivian Richards Stadium,2024,T20IS,...,9.000000,4.090909,13.200000,1.666667,3.666667,13.375000,5.631579,14.500000,1.60,3.800000
64030,2024-06-24,1415751,India,Australia,625383,JJ Bumrah,2,Darren Sammy National Cricket Stadium,2024,T20IS,...,8.285714,3.866667,12.857143,1.750000,3.750000,10.777778,5.105263,12.777778,1.80,3.800000
64031,2024-06-27,1415754,India,England,625383,JJ Bumrah,2,Providence Stadium,2024,T20IS,...,10.875000,4.578947,14.250000,1.600000,3.800000,10.875000,4.578947,14.375000,1.60,3.800000
64032,2024-06-29,1415755,India,South Africa,625383,JJ Bumrah,2,Kensington Oval,2024,T20IS,...,9.900000,4.569231,13.000000,1.666667,3.611111,11.625000,5.054348,14.125000,1.60,3.733333


# Final processing

In [42]:
min(updated_features_df.date), max(updated_features_df.date)

(Timestamp('2005-02-17 00:00:00'), Timestamp('2025-04-07 00:00:00'))

In [38]:
updated_features_df.career_avg.isnull().sum()

4691

In [39]:
def add_career_debut_column(df):
    """
    Adds a career_debut column to the dataframe.
    Sets value to 1 where career_avg is null (indicating league debut),
    and 0 for all other rows.

    Args:
        df: DataFrame containing batting or bowling features

    Returns:
        DataFrame with the additional career_debut column
    """
    # Create a copy to avoid modifying the original
    df = df.copy()

    # Create career_debut column based on whether career_avg is null
    df['career_debut'] = pd.isna(df['career_avg']).astype(int)

    print(f"Added career_debut column. Found {df['career_debut'].sum()} career debuts.")

    return df


# After creating your features dataframe
updated_features_df = add_career_debut_column(updated_features_df)

Added career_debut column. Found 4691 career debuts.


In [40]:
def convert_nan_to_zero(df):
    """
    Convert all NaN values in the dataframe to 0.

    Args:
        df: DataFrame containing features with potential NaN values

    Returns:
        DataFrame with NaN values replaced by 0
    """
    # Count NaN values before conversion
    nan_count_before = df.isna().sum().sum()

    # Make a copy to avoid modifying the original dataframe
    df_copy = df.copy()

    # Replace all NaN values with 0
    df_copy = df_copy.fillna(0)

    # Verify all NaNs have been replaced
    nan_count_after = df_copy.isna().sum().sum()

    print(f"Replaced {nan_count_before} NaN values with 0")
    print(f"Remaining NaN values: {nan_count_after}")

    return df_copy

# Usage:
updated_features_df = convert_nan_to_zero(updated_features_df)

Replaced 117320 NaN values with 0
Remaining NaN values: 0


In [41]:
updated_features_df.to_csv(r"D:\ED\FIFS_IPLungit\data\processed_data\all_league_bowl_features_dr.csv", index=False)